# TrustJob AI — Détection des offres d’emploi frauduleuses

**Projet complet** : comprendre le problème des fausses offres, préparer les données, comparer des méthodes, évaluer les résultats, puis réutiliser le meilleur modèle dans une interface web (HTML + `app.py`).

**Objectif** : à partir du texte et de quelques attributs d’une annonce, estimer si elle ressemble aux annonces frauduleuses du jeu d’entraînement. La cible `fraudulent` vaut 0 (légitime) ou 1 (frauduleuse). Le modèle donne une estimation statistique, pas une preuve juridique.

**Démarche** : exploration et visualisation (seaborn) → valeurs manquantes, doublons et valeurs aberrantes → prétraitement sans fuite de données → sélection de caractéristiques → description des 4 modèles → règle de référence → optimisation par validation croisée (4 plis) → matrices de confusion et tableaux récapitulatifs → robustesse → sauvegarde du pipeline utilisé par l’interface.

## 0 — Installation et données

**Bibliothèques** : pandas, numpy, matplotlib, seaborn, scikit-learn, joblib.

**Emplacement des fichiers** : placez ce notebook dans le même dossier que `job_postings_train.csv`, `job_postings_test.csv`, `job_postings_sample_submission.csv`, `app.py` et `index.html`. Le dossier `outputs/` créé ici est celui que lit `app.py`.

**Jeu de données** : EMSCAD (offres d’emploi, étiquette frauduleuse / légitime). Les CSV fournis contiennent 12 000 annonces d’entraînement et 18 colonnes ; le notebook affiche les dimensions réellement lues.

**Mode rapide** : `RAPIDE = True` réduit les grilles d’hyperparamètres pour tester le notebook en quelques minutes. Pour les résultats finaux, laisser `RAPIDE = False`.

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay,
                             precision_recall_curve, average_precision_score)

SEED = 42
TARGET = "fraudulent"
RAPIDE = False   # True = grilles réduites (test rapide) ; False = résultats finaux
N_JOBS = -1      # mettre 1 en cas d'erreur de mémoire
sns.set_theme(style="whitegrid")

CANDIDATE_DIRS = [Path.cwd(), Path.cwd() / "data", Path.cwd().parent, Path.home() / "Desktop" / "PROJET"]
DATA_DIR = next((d for d in CANDIDATE_DIRS if (d / "job_postings_train.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("job_postings_train.csv est introuvable : placez le notebook dans le dossier des CSV.")
OUTPUT_DIR = Path.cwd() / "outputs"          # dossier lu par app.py
OUTPUT_DIR.mkdir(exist_ok=True)

TEXT_FIELDS = ["title", "location", "department", "salary_range", "company_profile",
               "description", "requirements", "benefits", "employment_type",
               "required_experience", "required_education", "industry", "function"]
FLAG_FIELDS = ["telecommuting", "has_company_logo", "has_questions"]
print("Dossier données :", DATA_DIR.resolve())
print("Dossier résultats :", OUTPUT_DIR.resolve())

In [ ]:
train = pd.read_csv(DATA_DIR / "job_postings_train.csv")
test = pd.read_csv(DATA_DIR / "job_postings_test.csv")
sample_submission = pd.read_csv(DATA_DIR / "job_postings_sample_submission.csv")

print("Dimensions train :", train.shape)
print("Dimensions test sans étiquettes :", test.shape)
print("Colonnes :", train.columns.tolist())
assert TARGET in train.columns, f"La cible {TARGET!r} est absente."
assert set(TEXT_FIELDS + FLAG_FIELDS).issubset(train.columns), "Une colonne attendue manque."
display(train.head())
train.info()
display(train.describe(include="all").T)
for col in ["employment_type", "required_experience", "required_education", "industry", "function"]:
    print(col, ":", train[col].nunique(dropna=True), "valeurs non vides")
    display(train[col].fillna("Non renseigné").value_counts().head(8).rename("effectif").to_frame())

**Lecture du tableau.** Chaque ligne est une annonce. On distingue :
- la **cible** `fraudulent` (0/1) ;
- des **champs textuels** (titre, description, profil de l’entreprise, exigences, avantages…), qui portent l’essentiel de l’information ;
- des **catégories** (type d’emploi, expérience, formation, secteur, fonction) ;
- trois **indicateurs binaires** (`telecommuting`, `has_company_logo`, `has_questions`) ;
- l’identifiant `id`, qui ne sera **pas** utilisé comme prédicteur.

Beaucoup de champs sont très incomplets (par exemple `salary_range` et `department`, absents dans la majorité des annonces) : l’absence d’une information peut elle-même être un indice, d’où le traitement par chaîne vide plutôt que la suppression des lignes.

## 1 — Exploration : doublons, valeurs manquantes, visualisations

Les doublons exacts hors identifiant sont comptés puis retirés. Les valeurs textuelles manquantes seront remplacées par des chaînes vides et les indicateurs binaires manquants par 0 dans la préparation.

In [ ]:
feature_columns_for_duplicates = [c for c in train.columns if c != "id"]
duplicate_count = int(train.duplicated(subset=feature_columns_for_duplicates).sum())
print("Doublons exacts hors identifiant :", duplicate_count)
train = train.drop_duplicates(subset=feature_columns_for_duplicates).copy()

missing = train.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].rename("valeurs manquantes").to_frame())
y = train[TARGET].astype(int)
class_counts = y.value_counts().sort_index()
print("Effectifs :")
print(class_counts.rename(index={0: "Légitimes", 1: "Frauduleuses"}))
print(f"Taux frauduleux : {y.mean():.2%}")
print(f"Accuracy si l’on prédit toujours légitime : {(y == 0).mean():.2%} (aucune fraude détectée).")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=train, x=TARGET, hue=TARGET, legend=False, palette=["#4C78A8", "#E45756"], ax=ax)
ax.set(title="Répartition des annonces", xlabel="0 = légitime, 1 = frauduleuse", ylabel="Nombre d’annonces")
plt.tight_layout()
plt.show()

**Interprétation (répartition des classes)** : les annonces frauduleuses sont très minoritaires (environ 5 %). Un modèle qui répondrait toujours « légitime » aurait déjà environ 95 % d’accuracy sans détecter aucune fraude. L’accuracy seule est donc trompeuse : on suit surtout le **rappel**, la **précision** et le **F1 de la classe frauduleuse**.

In [ ]:
eda = train[[TARGET, "description"]].copy()
eda["longueur_mots"] = eda["description"].fillna("").astype(str).str.split().str.len()
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=eda, x="longueur_mots", hue=TARGET, bins=35, stat="density",
             common_norm=False, element="step", ax=ax)
ax.set(title="Longueur des descriptions par classe", xlabel="Nombre de mots", ylabel="Densité")
plt.tight_layout()
plt.show()
print(eda.groupby(TARGET)["longueur_mots"].describe()[["count", "mean", "50%", "max"]].round(1))

**Interprétation (longueur des descriptions)** : les deux classes ont des distributions proches, mais les annonces frauduleuses (orange) ont tendance à être **plus courtes** (pic plus marqué sous une centaine de mots). La longueur peut servir d’indice, pas de preuve ; le graphique est descriptif et ne démontre pas de causalité.

In [ ]:
numeric_for_corr = train[FLAG_FIELDS + [TARGET]].apply(pd.to_numeric, errors="coerce")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(numeric_for_corr.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
ax.set_title("Corrélations des indicateurs binaires et de la cible")
plt.tight_layout()
plt.show()

**Interprétation (corrélations)** : `has_company_logo` est la variable la plus liée à la cible (corrélation d’environ **-0,26** : les annonces avec logo sont moins souvent frauduleuses) ; `has_questions` est faiblement liée (environ -0,09) et `telecommuting` presque pas (environ 0,04). Les indicateurs sont peu corrélés entre eux (le plus fort : logo / questions, environ 0,23). Ce sont des corrélations, pas des relations de cause à effet.

In [ ]:
ind = train[FLAG_FIELDS].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
ind["profil_entreprise_renseigné"] = train["company_profile"].notna().astype(int)
ind[TARGET] = train[TARGET].astype(int)
cols = FLAG_FIELDS + ["profil_entreprise_renseigné"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
for col, ax in zip(cols, axes):
    rate = ind.groupby(col)[TARGET].mean().reset_index()
    sns.barplot(data=rate, x=col, y=TARGET, hue=col, legend=False, palette="Set2", ax=ax)
    ax.set(title=col, xlabel="0 = non, 1 = oui", ylabel="Taux de fraude")
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1%}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()
for col in ["has_company_logo", "profil_entreprise_renseigné"]:
    r = ind.groupby(col)[TARGET].mean()
    if 0 in r.index and 1 in r.index and r[1] > 0:
        print(f"{col} : taux de fraude {r[0]:.1%} si 0 contre {r[1]:.1%} si 1 (x{r[0] / r[1]:.1f}).")

**Interprétation (taux de fraude par indicateur)** : on compare le taux de fraude selon la modalité de chaque indicateur. Une annonce **sans logo** ou **sans profil d’entreprise** a un taux de fraude plus élevé que le taux moyen (valeurs affichées ci-dessus). Ces indices justifient de conserver les indicateurs binaires dans le modèle, en complément du texte.

In [ ]:
q1, q3 = eda["longueur_mots"].quantile([0.25, 0.75])
iqr = q3 - q1
borne_sup = q3 + 1.5 * iqr
eda["atypique"] = eda["longueur_mots"] > borne_sup
nb_atypiques = int(eda["atypique"].sum())
fig, ax = plt.subplots(figsize=(8, 3))
sns.boxplot(x=eda["longueur_mots"], color="#72B7B2", ax=ax)
ax.set(title="Valeurs atypiques de longueur de description", xlabel="Nombre de mots")
plt.tight_layout()
plt.show()
print(f"IQR : Q1={q1:.0f}, Q3={q3:.0f}, borne supérieure={borne_sup:.0f}; annonces au-dessus={nb_atypiques}.")
print("Part d’annonces atypiques par classe :")
print(eda.groupby(TARGET)["atypique"].mean().rename("part atypiques").map("{:.1%}".format))
print("Décision : les annonces longues sont conservées, car leur longueur n’est pas nécessairement une erreur.")

**Interprétation (valeurs aberrantes, règle de l’IQR)** : le boxplot et la borne Q3 + 1,5 × IQR font ressortir quelques descriptions très longues. Elles sont **atypiques mais pas erronées** (une annonce détaillée peut être réelle) et ne sont pas concentrées dans une seule classe : elles sont donc **conservées**. De plus, le TF-IDF et le `sublinear_tf` limitent déjà l’influence des textes très longs. Les indicateurs binaires n’ont pas de valeurs aberrantes possibles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for col, ax in zip(["employment_type", "required_experience"], axes):
    counts = train[col].fillna("Non renseigné").value_counts().head(10)
    sns.barplot(x=counts.values, y=counts.index, hue=counts.index, legend=False, palette="viridis", ax=ax)
    ax.set(title=f"Catégories fréquentes — {col}", xlabel="Nombre d’annonces", ylabel="")
plt.tight_layout()
plt.show()

**Interprétation (catégories fréquentes)** : quelques modalités dominent (temps plein pour le type d’emploi) et les champs non renseignés sont nombreux. Ils sont remplacés par une chaîne vide lors de la préparation pour ne pas perdre les lignes.

In [ ]:
rep = train["description"].duplicated(keep=False) & train["description"].notna()
print(f"Descriptions uniques : {train['description'].nunique()} sur {train['description'].notna().sum()} non vides.")
print(f"Annonces dont la description apparaît plusieurs fois : {int(rep.sum())} ({rep.mean():.1%}).")
print(f"Taux de fraude — description répétée : {train.loc[rep, TARGET].mean():.2%} | description unique : {train.loc[~rep, TARGET].mean():.2%}")

**Point de vigilance (quasi-doublons)** : une même description peut apparaître dans plusieurs annonces (republications). Avec un découpage aléatoire, des annonces quasi identiques peuvent se retrouver à la fois en apprentissage et en test, ce qui rend les scores un peu **optimistes**. Cet effet est mesuré dans la section 7.

## 2 — Prétraitement et sélection de caractéristiques (sans fuite)

- **Prétraitement** : les champs textuels sont concaténés en un seul texte (valeurs manquantes → chaîne vide), puis transformés en **TF-IDF** (unigrammes et bigrammes, mots vides anglais retirés). Les trois indicateurs binaires sont conservés (manquants → 0).
- **Sélection de caractéristiques** : `SelectKBest` avec le test du **χ²** garde les `k` termes les plus liés à la cible. La valeur de `k` est choisie par validation croisée (section 5).
- **Pas de scaling** : la matrice TF-IDF est creuse et déjà normalisée ; centrer-réduire la détruirait la sparsité.

**Anti-fuite (data leakage)** : toutes les transformations sont dans un `Pipeline` scikit-learn. Le découpage apprentissage / test interne (80 / 20, stratifié) est fait avant tout ajustement ; dans la validation croisée, `TfidfVectorizer` et `SelectKBest` sont **ajustés uniquement sur les plis d’apprentissage**.

In [ ]:
def prepare(frame):
    out = pd.DataFrame(index=frame.index)
    out["text"] = frame[TEXT_FIELDS].fillna("").astype(str).agg(" ".join, axis=1)
    for col in FLAG_FIELDS:
        out[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0).astype(float)
    return out

X = prepare(train)
y = train[TARGET].astype(int)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Apprentissage :", X_train.shape, "| test interne :", X_holdout.shape)
print("Taux fraude train :", f"{y_train.mean():.2%}")
print("Taux fraude test interne :", f"{y_holdout.mean():.2%}", f"({int(y_holdout.sum())} fraudes)")

def make_pipeline(estimator, k=3000):
    text_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2,
                                  max_features=20000, sublinear_tf=True)),
        ("select", SelectKBest(score_func=chi2, k=k)),
    ])
    features = ColumnTransformer([
        ("text", text_pipeline, "text"),
        ("flags", "passthrough", FLAG_FIELDS),
    ], sparse_threshold=1.0)
    return Pipeline([("features", features), ("model", estimator)])

In [ ]:
# Illustration : termes les mieux classés par le χ² (calcul sur l'apprentissage uniquement)
tfidf_demo = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2,
                             max_features=20000, sublinear_tf=True)
M = tfidf_demo.fit_transform(X_train["text"])
scores, _ = chi2(M, y_train)
top = (pd.DataFrame({"terme": tfidf_demo.get_feature_names_out(), "chi2": scores})
       .dropna().nlargest(15, "chi2"))
print("Nombre de termes TF-IDF candidats :", M.shape[1])
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=top, x="chi2", y="terme", hue="terme", legend=False, palette="rocket", ax=ax)
ax.set(title="15 termes les plus liés à la fraude (score χ²)", xlabel="Score χ²", ylabel="")
plt.tight_layout()
plt.show()

**Interprétation (sélection)** : parmi les dizaines de milliers de termes candidats, seuls les plus discriminants sont gardés. Ce graphique est **illustratif** : dans la validation croisée, la sélection est refaite à chaque pli pour éviter toute fuite. Réduire la dimension accélère l’entraînement et limite le sur-apprentissage.

## 3 — Description des modèles

Les quatre modèles reçoivent la **même représentation** :

```
texte brut ──► TF-IDF (1-2 grammes) ──► SelectKBest χ² (k termes) ──┐
                                                                    ├──► classifieur ──► 0 / 1
3 indicateurs binaires (logo, questions, télétravail) ──────────────┘
```

### 3.1 Régression logistique
- **Définition** : modèle linéaire qui estime la probabilité qu’une annonce soit frauduleuse : P(fraude | x) = σ(w·x + b), avec σ(z) = 1 / (1 + e^(−z)).
- **Étapes** : (1) l’annonce devient un vecteur x ; (2) on calcule le score linéaire z = w·x + b ; (3) la sigmoïde le transforme en probabilité ; (4) l’apprentissage ajuste w et b en minimisant la perte logistique avec une pénalité L2 ; (5) on classe « frauduleuse » si la probabilité dépasse 0,5.
- **Hyperparamètre optimisé** : `C` (inverse de la régularisation : petit `C` = modèle plus simple). `class_weight="balanced"` compense la rareté des fraudes.
- **Atouts / limites** : rapide, interprétable (poids par terme), mais frontière strictement linéaire.

### 3.2 SVM linéaire
- **Définition** : cherche l’hyperplan w·x + b = 0 qui sépare les deux classes avec la **marge maximale** ; seuls les points proches de la marge (vecteurs de support) le déterminent.
- **Étapes** : (1) vecteur x ; (2) apprentissage : minimiser ‖w‖² + C · Σ perte « charnière » (hinge) ; (3) prédiction par le signe de w·x + b.
- **Hyperparamètre optimisé** : `C` (compromis entre marge large et erreurs tolérées). `class_weight="balanced"`.
- **Atouts / limites** : très efficace en grande dimension creuse (cas du texte) ; ne fournit pas de probabilité native (on utilise la distance à l’hyperplan, `decision_function`).

### 3.3 K plus proches voisins (KNN)
- **Définition** : une annonce reçoit la classe majoritaire de ses `k` voisins les plus proches dans l’espace TF-IDF (distance euclidienne sur des vecteurs normalisés, proche d’une similarité cosinus).
- **Étapes** : (1) mémoriser les annonces d’apprentissage ; (2) pour une nouvelle annonce, calculer la distance à toutes les autres ; (3) retenir les `k` plus proches ; (4) vote (uniforme ou pondéré par la distance).
- **Hyperparamètres optimisés** : `n_neighbors` et `weights`.
- **Atouts / limites** : aucun apprentissage, simple ; mais prédiction lente, sensible au déséquilibre (pas de pondération des classes) et à la dimension.

### 3.4 Random Forest
- **Définition** : ensemble de nombreux arbres de décision, chacun entraîné sur un échantillon bootstrap des annonces et un sous-ensemble aléatoire de termes ; la prédiction est le **vote majoritaire**.
- **Étapes** : (1) tirer un échantillon bootstrap ; (2) construire un arbre en testant à chaque nœud un sous-ensemble aléatoire de variables ; (3) répéter `n_estimators` fois ; (4) agréger les votes.
- **Hyperparamètres optimisés** : `n_estimators` et `max_depth`. `class_weight="balanced"`.
- **Atouts / limites** : robuste au bruit, capture des interactions ; plus lourd et en général moins adapté que les modèles linéaires au texte creux de grande dimension.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import NearestNeighbors
from matplotlib.patches import Circle

Xt, yt = make_blobs(n_samples=60, centers=[(-1.5, -1.0), (1.5, 1.0)], cluster_std=1.1, random_state=SEED)
palette = {0: "#4C78A8", 1: "#E45756"}
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Régression logistique : la sigmoïde
ax = axes[0, 0]
z = np.linspace(-6, 6, 200)
ax.plot(z, 1 / (1 + np.exp(-z)), color="#333", lw=2)
ax.axhline(0.5, ls="--", color="grey"); ax.axvline(0, ls="--", color="grey")
ax.fill_between(z, 0, 1, where=z > 0, color=palette[1], alpha=0.08)
ax.fill_between(z, 0, 1, where=z <= 0, color=palette[0], alpha=0.08)
ax.set(title="Régression logistique : P(fraude) = σ(w·x + b)", xlabel="score linéaire w·x + b", ylabel="probabilité")
ax.text(-5.5, 0.85, "légitime", color=palette[0]); ax.text(3, 0.15, "frauduleuse", color=palette[1])

# SVM : hyperplan et marges
ax = axes[0, 1]
svm_demo = LinearSVC(C=1.0, random_state=SEED).fit(Xt, yt)
gx, gy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
Z = svm_demo.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
ax.contour(gx, gy, Z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
sns.scatterplot(x=Xt[:, 0], y=Xt[:, 1], hue=yt, palette=palette, legend=False, ax=ax)
ax.set(title="SVM linéaire : hyperplan (trait plein) et marges (pointillés)", xlabel="variable 1", ylabel="variable 2")

# KNN : voisinage
ax = axes[1, 0]
q = np.array([[0.3, 0.2]])
nn = NearestNeighbors(n_neighbors=5).fit(Xt)
dist, idx = nn.kneighbors(q)
sns.scatterplot(x=Xt[:, 0], y=Xt[:, 1], hue=yt, palette=palette, legend=False, ax=ax)
ax.scatter(Xt[idx[0], 0], Xt[idx[0], 1], s=140, facecolors="none", edgecolors="k")
ax.scatter(q[:, 0], q[:, 1], marker="*", s=250, color="gold", edgecolors="k", zorder=5)
ax.add_patch(Circle(q[0], dist.max(), fill=False, ls="--", color="grey"))
ax.set_aspect("equal", adjustable="datalim")
ax.set(title="KNN (k = 5) : l'étoile prend la classe majoritaire de ses voisins", xlabel="variable 1", ylabel="variable 2")

# Random Forest : un arbre de la forêt
ax = axes[1, 1]
tree_demo = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(Xt, yt)
plot_tree(tree_demo, filled=True, feature_names=["x1", "x2"], class_names=["légitime", "frauduleuse"],
          impurity=False, ax=ax)
ax.set_title("Random Forest : un arbre parmi n_estimators (vote majoritaire)")
plt.tight_layout()
plt.show()

**Lecture des figures** : (1) la sigmoïde convertit un score linéaire en probabilité, seuil à 0,5 ; (2) le SVM place la frontière au milieu de la marge maximale ; (3) le KNN regarde les 5 points les plus proches de l’annonce à classer ; (4) la forêt agrège de nombreux arbres comme celui-ci. Ce sont des illustrations en 2 dimensions sur données jouets ; sur nos annonces, l’espace compte plusieurs milliers de dimensions.

## 4 — Méthode traditionnelle de comparaison (règle de mots-clés)

Une règle fixe signale une annonce si le texte contient des expressions comme « upfront fee », « bank account » ou « frais de dossier ». Elle n’apprend rien des données : elle sert de **point de repère**. Le test interne est le même que pour les modèles.

In [ ]:
RULE_TERMS = [
    "upfront fee", "processing fee", "application fee", "pay a fee",
    "wire transfer", "western union", "money order", "bank account",
    "social security number", "passport", "frais de dossier",
    "paiement préalable", "virement bancaire", "coordonnées bancaires"
]
def traditional_rule(text):
    text = str(text).lower()
    return int(any(term in text for term in RULE_TERMS))

traditional_text = train.loc[X_holdout.index, TEXT_FIELDS].fillna("").astype(str).agg(" ".join, axis=1)
traditional_pred = traditional_text.map(traditional_rule).to_numpy()
traditional_metrics = {
    "Modèle": "Règle traditionnelle",
    "Meilleurs paramètres": "Liste fixe de mots-clés",
    "F1 CV (4 plis)": np.nan,
    "Écart-type CV": np.nan,
    "Accuracy": accuracy_score(y_holdout, traditional_pred),
    "Précision": precision_score(y_holdout, traditional_pred, zero_division=0),
    "Rappel": recall_score(y_holdout, traditional_pred, zero_division=0),
    "F1 test interne": f1_score(y_holdout, traditional_pred, zero_division=0),
}
display(pd.DataFrame([traditional_metrics]))
print(f"Part d'annonces signalées par la règle : {traditional_pred.mean():.1%}")
print(classification_report(y_holdout, traditional_pred, target_names=["Légitime", "Frauduleuse"], zero_division=0))

**Interprétation** : la règle signale surtout des annonces légitimes (beaucoup d’offres réelles parlent de « bank account » ou de « passport » dans des contextes banals) et rate presque toutes les fraudes. Son accuracy est même **inférieure** à celle du modèle « toujours légitime ». Les expressions suspectes ne suffisent donc pas : le contexte compte, ce que les modèles apprennent.

## 5 — Choix du nombre de caractéristiques `k` (validation croisée, 4 plis)

Avant de comparer les modèles, on règle le nombre de termes gardés par `SelectKBest`. Un SVM linéaire sert de modèle de référence (rapide) ; le `k` retenu est ensuite utilisé pour les 4 modèles. La CV est stratifiée à 4 plis et le critère est le **F1 de la classe frauduleuse**.

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
k_values = [500, 3000] if RAPIDE else [500, 1000, 3000, 5000, "all"]
k_search = GridSearchCV(
    make_pipeline(LinearSVC(class_weight="balanced", C=1.0, max_iter=5000, random_state=SEED)),
    param_grid={"features__text__select__k": k_values},
    scoring="f1", cv=cv, n_jobs=N_JOBS, refit=False,
)
k_search.fit(X_train, y_train)
k_res = pd.DataFrame({
    "k": [str(k) for k in k_values],
    "F1 CV": k_search.cv_results_["mean_test_score"],
    "écart-type": k_search.cv_results_["std_test_score"],
})
K_BEST = k_values[int(np.argmax(k_res["F1 CV"]))]
display(k_res)
fig, ax = plt.subplots(figsize=(7, 4))
sns.pointplot(data=k_res, x="k", y="F1 CV", color="#4C78A8", ax=ax)
ax.errorbar(range(len(k_res)), k_res["F1 CV"], yerr=k_res["écart-type"], fmt="none", ecolor="grey", capsize=3)
ax.set(title="F1 en validation croisée selon k (SVM linéaire)", xlabel="k (nombre de termes gardés)", ylabel="F1 CV")
plt.tight_layout()
plt.show()
print("k retenu :", K_BEST)

**Interprétation** : on retient le `k` qui maximise le F1 moyen en CV ; les barres d’erreur montrent la variabilité entre plis. Si plusieurs valeurs sont équivalentes, la plus petite est préférable (modèle plus léger).

## 6 — Apprentissage des 4 modèles et optimisation (GridSearchCV, CV = 4)

Pour chaque modèle, `GridSearchCV` explore une grille d’hyperparamètres avec la même validation croisée stratifiée à 4 plis, critère **F1 de la classe frauduleuse**. Le pipeline complet (TF-IDF, sélection, classifieur) est ré-ajusté à chaque pli. Le meilleur modèle de chaque grille est ensuite évalué sur le **test interne**, jamais utilisé pour le réglage.

In [ ]:
def pick(full, fast):
    return fast if RAPIDE else full

model_specs = [
    ("Régression logistique",
     LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear"),
     {"model__C": pick([0.1, 0.5, 1, 2, 5, 10], [0.5, 5])}),
    ("KNN",
     KNeighborsClassifier(algorithm="brute"),
     {"model__n_neighbors": pick([3, 5, 7, 11], [3, 7]), "model__weights": ["uniform", "distance"]}),
    ("SVM linéaire",
     LinearSVC(class_weight="balanced", max_iter=5000, random_state=SEED),
     {"model__C": pick([0.1, 0.5, 1, 2, 5, 10], [0.5, 5])}),
    ("Random Forest",
     RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=1),
     {"model__n_estimators": pick([100, 200], [50]), "model__max_depth": pick([20, 40, None], [20, None])}),
]
searches, grids, rows = {}, {}, []
for model_name, estimator, param_grid in model_specs:
    print(f"Optimisation : {model_name}")
    search = GridSearchCV(
        estimator=make_pipeline(estimator, k=K_BEST), param_grid=param_grid,
        scoring="f1", cv=cv, n_jobs=N_JOBS, refit=True, verbose=1,
    )
    search.fit(X_train, y_train)
    pred = search.predict(X_holdout)
    i = search.best_index_
    rows.append({
        "Modèle": model_name,
        "Meilleurs paramètres": search.best_params_,
        "F1 CV (4 plis)": search.best_score_,
        "Écart-type CV": search.cv_results_["std_test_score"][i],
        "Accuracy": accuracy_score(y_holdout, pred),
        "Précision": precision_score(y_holdout, pred, zero_division=0),
        "Rappel": recall_score(y_holdout, pred, zero_division=0),
        "F1 test interne": f1_score(y_holdout, pred, zero_division=0),
        "Matrice": confusion_matrix(y_holdout, pred, labels=[0, 1]),
        "_pred": pred,
    })
    searches[model_name] = search
    grids[model_name] = param_grid
    print("Meilleurs paramètres :", search.best_params_)
    print(classification_report(y_holdout, pred, target_names=["Légitime", "Frauduleuse"], zero_division=0))

In [ ]:
# Détail des grilles (CV) et contrôle : un optimum au bord de la grille doit alerter
for name, s in searches.items():
    detail = (pd.DataFrame(s.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
              .sort_values("rank_test_score").rename(columns={
                  "mean_test_score": "F1 CV", "std_test_score": "écart-type", "rank_test_score": "rang"}))
    print(f"\n{name} — 5 meilleures combinaisons")
    display(detail.head(5).reset_index(drop=True))
    for p, v in s.best_params_.items():
        vals = [x for x in grids[name][p] if isinstance(x, (int, float)) and not isinstance(x, bool)]
        if isinstance(v, (int, float)) and len(vals) > 2 and v in (min(vals), max(vals)):
            print(f"  ⚠ {p} = {v} est au bord de la grille {vals} : élargir la grille pourrait améliorer le résultat.")

## 7 — Résultats : tableaux, matrices de confusion, courbes et robustesse

Le modèle retenu est celui qui a le meilleur **F1 en validation croisée** (et non sur le test interne, pour que le test reste une évaluation indépendante).

In [ ]:
DROP_KEYS = {"Matrice", "_pred"}
ml_rows = [{k: v for k, v in row.items() if k not in DROP_KEYS} for row in rows]
comparison = pd.DataFrame([traditional_metrics] + ml_rows)
display(comparison.sort_values("F1 CV (4 plis)", ascending=False, na_position="last").reset_index(drop=True))

best_row = max(rows, key=lambda r: r["F1 CV (4 plis)"])
best_name = best_row["Modèle"]
best_by_test = max(rows, key=lambda r: r["F1 test interne"])["Modèle"]
print(f"Modèle retenu (meilleur F1 en CV) : {best_name}")
print(f"F1 test interne={best_row['F1 test interne']:.3f} | précision={best_row['Précision']:.3f} | rappel={best_row['Rappel']:.3f}")
if best_by_test != best_name:
    print(f"⚠ Le meilleur F1 sur le test interne est celui de {best_by_test} : index.html trie par F1 test interne et "
          f"affichera ce nom comme « modèle retenu », alors que le pipeline sauvegardé est {best_name}.")

In [ ]:
metric_columns = ["Accuracy", "Précision", "Rappel", "F1 test interne"]
plot_data = comparison.melt(id_vars="Modèle", value_vars=metric_columns, var_name="Métrique", value_name="Score")
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_data, x="Modèle", y="Score", hue="Métrique", ax=ax)
ax.set(title="Comparaison des modèles sur le test interne", ylim=(0, 1), xlabel="", ylabel="Score")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

**Interprétation** : on compare Accuracy, Précision, Rappel et F1 sur le test interne. Les accuracies sont proches (le déséquilibre des classes les écrase vers le haut) ; ce sont le **rappel** et le **F1** qui distinguent les modèles. Un modèle très précis mais au rappel faible (typiquement le KNN) laisse passer des fraudes ; un modèle au rappel élevé mais à la précision faible (typiquement la régression logistique) multiplie les fausses alertes. La règle traditionnelle est très en dessous de tous les modèles.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, row in zip(axes.ravel(), rows):
    ConfusionMatrixDisplay(row["Matrice"], display_labels=["Légitime", "Frauduleuse"]).plot(
        ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(row["Modèle"])
plt.tight_layout()
plt.show()

print(f"Test interne : {len(y_holdout)} annonces dont {int(y_holdout.sum())} frauduleuses.\n")
for row in rows:
    (vn, fp), (fn, vp) = row["Matrice"]
    print(f"{row['Modèle']:<22} VN={vn:<5} FP={fp:<4} FN={fn:<4} VP={vp:<4} -> "
          f"{vp}/{vp + fn} fraudes détectées ({vp / (vp + fn):.0%}), {fp} fausses alertes")

**Lecture des matrices de confusion** : VN = légitimes bien classées, FP = légitimes signalées à tort (fausse alerte), FN = **fraudes manquées**, VP = fraudes détectées. En détection de fraude, un FN est en général plus coûteux qu’un FP ; on cherche donc un bon compromis rappel / précision, pas seulement peu d’erreurs au total. Les chiffres ci-dessus se lisent modèle par modèle.

In [ ]:
def scores_of(est, Xh):
    return est.predict_proba(Xh)[:, 1] if hasattr(est, "predict_proba") else est.decision_function(Xh)

fig, ax = plt.subplots(figsize=(7, 5))
for name, s in searches.items():
    sc = scores_of(s.best_estimator_, X_holdout)
    p, r, _ = precision_recall_curve(y_holdout, sc)
    ax.plot(r, p, label=f"{name} (AP = {average_precision_score(y_holdout, sc):.2f})")
ax.axhline(y_holdout.mean(), ls="--", color="grey", label="hasard")
ax.set(title="Courbes précision-rappel (test interne)", xlabel="Rappel", ylabel="Précision", ylim=(0, 1.02))
ax.legend()
plt.tight_layout()
plt.show()

**Interprétation** : chaque courbe montre la précision obtenue pour un rappel donné quand on fait varier le seuil d’alerte ; plus elle est proche du coin supérieur droit, mieux c’est. L’AP (average precision) résume la courbe. C’est le bon outil quand les classes sont déséquilibrées, et elle permet de choisir un seuil selon le coût relatif d’une fraude manquée et d’une fausse alerte.

In [ ]:
# Robustesse 1 : annonces du test interne dont la description est absente de l'apprentissage
seen = train.loc[X_holdout.index, "description"].isin(train.loc[X_train.index, "description"]).to_numpy()
mask = ~seen
yh = y_holdout.to_numpy()
ph = best_row["_pred"]
print(f"{best_name} — annonces du test interne dont la description figure aussi dans l'apprentissage : {seen.mean():.1%}")
if mask.sum() > 0 and yh[mask].sum() > 0:
    print(f"Sur les {int(mask.sum())} annonces à description inédite ({int(yh[mask].sum())} fraudes) : "
          f"F1={f1_score(yh[mask], ph[mask], zero_division=0):.3f} | "
          f"précision={precision_score(yh[mask], ph[mask], zero_division=0):.3f} | "
          f"rappel={recall_score(yh[mask], ph[mask], zero_division=0):.3f}")
    print(f"Rappel : F1 sur l'ensemble du test interne = {best_row['F1 test interne']:.3f}")

# Robustesse 2 : intervalle de confiance bootstrap du F1 (le test interne contient peu de fraudes)
rng = np.random.default_rng(SEED)
boots = []
for _ in range(1000):
    idx = rng.integers(0, len(yh), len(yh))
    boots.append(f1_score(yh[idx], ph[idx], zero_division=0))
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"F1 test interne de {best_name} : {best_row['F1 test interne']:.3f} (IC 95 % bootstrap : {lo:.3f} – {hi:.3f})")

**Interprétation de la robustesse** : (1) si le score baisse nettement sur les annonces à description inédite, les quasi-doublons gonflent les résultats ; (2) l’intervalle de confiance montre l’incertitude liée au faible nombre de fraudes dans le test interne : les petites différences de F1 entre deux modèles ne sont pas forcément significatives.

### Rappel — métriques à privilégier (classe frauduleuse)
- **Rappel** : part des fraudes réellement détectées (éviter les faux négatifs).
- **Précision** : parmi les alertes, part de vraies fraudes (limiter les faux positifs).
- **F1** : compromis entre précision et rappel, adapté aux classes déséquilibrées.
- **Accuracy** : à interpréter avec prudence (la classe légitime domine).

## 8 — Sauvegarde du pipeline et prédictions

Le modèle retenu est ré-entraîné sur **toutes** les annonces étiquetées. Le fichier joblib contient le pipeline complet (TF-IDF, sélection, indicateurs, classifieur). Il est lu par `app.py` ; le tableau de comparaison est lu par `index.html` (onglet « Performance des modèles »).

In [ ]:
best_pipeline = clone(searches[best_name].best_estimator_)
best_pipeline.fit(X, y)
MODEL_PATH = OUTPUT_DIR / "trustjob_pipeline.joblib"
joblib.dump(best_pipeline, MODEL_PATH, compress=3)
comparison.to_csv(OUTPUT_DIR / "comparaison_modeles_trustjob.csv", index=False)
metrics_json = [
    {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in row.items() if k not in DROP_KEYS}
    for row in rows
]
(OUTPUT_DIR / "resultats_trustjob.json").write_text(
    json.dumps({"seed": SEED, "k_selectkbest": str(K_BEST), "train_shape": list(train.shape),
                "holdout_rows": len(X_holdout), "best_model": best_name, "results": metrics_json},
               ensure_ascii=False, indent=2, default=str),
    encoding="utf-8")
print("Pipeline complet sauvegardé :", MODEL_PATH.resolve())
print("Tableau de comparaison :", (OUTPUT_DIR / "comparaison_modeles_trustjob.csv").resolve())

In [ ]:
test_X = prepare(test)
test_predictions = best_pipeline.predict(test_X).astype(int)
submission = pd.DataFrame({"id": test["id"].values, TARGET: test_predictions})
if list(sample_submission.columns) == list(submission.columns):
    submission = submission[sample_submission.columns]
submission_path = OUTPUT_DIR / "submission_predictions.csv"
submission.to_csv(submission_path, index=False)
print("Prédictions créées :", submission_path.resolve(), "| lignes :", len(submission))
print(f"Part d'annonces prédites frauduleuses : {submission[TARGET].mean():.1%}")
display(submission.head())

**Le fichier test n’a pas d’étiquettes** : il ne permet pas de calculer de métriques, seulement de produire des prédictions au format de soumission.

### Application de démonstration (facultatif)
Une interface web utilise le pipeline sauvegardé :
- `app.py` : serveur Python standard (aucun framework) qui charge `outputs/trustjob_pipeline.joblib` et expose `POST /api/predict` (texte + 3 indicateurs → prédiction, score de risque, expressions suspectes) et `GET /api/metrics` (tableau de comparaison) ;
- `index.html` : interface (vérification d’une annonce, analyse par lot, performance des modèles, historique).

Lancement : `python app.py` dans le dossier du projet, puis ouvrir http://localhost:5000. Le score de risque affiché est un **indice** dérivé de la distance à l’hyperplan du SVM, pas une probabilité calibrée.

## 9 — Annexe facultative : clustering exploratoire (K-Means et CAH)

Le clustering regroupe des annonces semblables **sans utiliser la cible** ; il ne prédit donc pas la fraude. On utilise un échantillon et une réduction TF-IDF + SVD pour limiter le temps de calcul. Le nombre de groupes est choisi avec le score de silhouette ; la cible n’est ajoutée qu’après coup pour interpréter les groupes.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

cluster_sample = train.sample(n=min(800, len(train)), random_state=SEED).copy()
cluster_text = cluster_sample[TEXT_FIELDS].fillna("").astype(str).agg(" ".join, axis=1)
cluster_matrix = TfidfVectorizer(stop_words="english", max_features=10000, min_df=2).fit_transform(cluster_text)
cluster_vectors = TruncatedSVD(n_components=min(50, cluster_matrix.shape[1] - 1), random_state=SEED).fit_transform(cluster_matrix)

sil = {k: silhouette_score(cluster_vectors, KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(cluster_vectors))
       for k in range(2, 6)}
K_CL = max(sil, key=sil.get)
print("Silhouette par nombre de groupes :", {k: round(v, 3) for k, v in sil.items()}, "-> k =", K_CL)

cluster_sample["groupe_kmeans"] = KMeans(n_clusters=K_CL, random_state=SEED, n_init=10).fit_predict(cluster_vectors)
cluster_sample["groupe_cah"] = AgglomerativeClustering(n_clusters=K_CL).fit_predict(cluster_vectors)
for group_col in ["groupe_kmeans", "groupe_cah"]:
    summary = (cluster_sample.groupby(group_col)[TARGET].agg(["count", "mean"])
               .rename(columns={"count": "annonces", "mean": "taux_fraude"}))
    display(summary)
    sns.countplot(data=cluster_sample, x=group_col, hue=TARGET, palette=["#4C78A8", "#E45756"])
    plt.title(f"Classes observées après regroupement — {group_col}")
    plt.xlabel("Groupe découvert sans utiliser la cible")
    plt.ylabel("Nombre d’annonces")
    plt.tight_layout()
    plt.show()

**Interprétation** : les groupes découverts sans la cible sont en général **déséquilibrés** et contiennent les deux classes ; K-Means et CAH peuvent donner des partitions voisines. Le clustering aide à explorer les types d’annonces mais **ne remplace pas** les classifieurs supervisés, qui utilisent l’étiquette.

In [ ]:
best_cv = best_row["F1 CV (4 plis)"]
print("RÉSUMÉ FINAL")
print(f"- Modèle retenu : {best_name} avec {best_row['Meilleurs paramètres']} et k = {K_BEST}")
print(f"- F1 en validation croisée (4 plis) : {best_cv:.3f} ± {best_row['Écart-type CV']:.3f}")
print(f"- Test interne : F1={best_row['F1 test interne']:.3f} | rappel={best_row['Rappel']:.3f} | "
      f"précision={best_row['Précision']:.3f} | accuracy={best_row['Accuracy']:.3f}")
(vn, fp), (fn, vp) = best_row["Matrice"]
print(f"- Fraudes manquées : {fn}/{vp + fn} ({fn / (vp + fn):.0%}) ; fausses alertes : {fp}")
print(f"- Règle de mots-clés (référence) : F1={traditional_metrics['F1 test interne']:.3f}")

## Conclusion et limites

**Ce que le projet réalise** :
- **Problème** : aider à repérer les annonces d’emploi potentiellement frauduleuses.
- **Données** : exploration et visualisation avec seaborn, vérification des doublons, des valeurs manquantes et des valeurs aberrantes (IQR / boxplot).
- **Méthodes** : une règle traditionnelle de mots-clés comparée à 4 modèles de machine learning (régression logistique, KNN, SVM linéaire, Random Forest), avec prétraitement TF-IDF sans fuite de données, sélection de caractéristiques (`SelectKBest` χ², `k` choisi par CV) et optimisation des hyperparamètres par validation croisée stratifiée à 4 plis (critère F1).
- **Résultats** : matrices de confusion, tableaux récapitulatifs, courbes précision-rappel, robustesse (quasi-doublons, intervalle de confiance) ; chiffres du modèle retenu dans le **résumé final** ci-dessus. En détection de fraude, rappel et F1 priment sur l’accuracy.
- **Au-delà du modèle** : le pipeline est sauvegardé et utilisé par l’interface web (`app.py` + `index.html`).

**Limites** :
- corpus et période limités ; les fraudes évoluent dans le temps ;
- peu de fraudes dans le test interne (résultats incertains, voir l’intervalle de confiance) ;
- quasi-doublons possibles entre apprentissage et test (scores potentiellement optimistes) ;
- fichier test sans étiquettes ;
- grilles d’hyperparamètres volontairement compactes pour limiter le temps de calcul ;
- le score de risque de l’interface est un indice, pas une probabilité calibrée.

**Pistes d’amélioration** : modèles de langage plus récents, découpage par groupes d’annonces identiques, ajout de caractéristiques (présence d’e-mail ou de lien, salaire renseigné), vérifications externes (existence légale de l’entreprise), suivi des performances dans le temps et réglage du seuil selon le coût faux négatif / faux positif.